# 04 · Graph

Block 4 check: the whole LangGraph graph, three full runs on the twin — normal, no particles, failure.

Remember: on the twin, tuning ends at random points (its image does not change with z / C1 / A1); the runs check the wiring, the routing and the records.

**Before running (twin):** `uv run python scripts/twin.py start`.

In [ ]:
TARGET = "twin"   # "twin" or "real"

BASE = {
    "target": TARGET,
    "target_xy_m": (5e-9, 0.0),
    "tuning_fov_m": 40e-9,
    "tuning": {"z": {"max_iterations": 8}},   # shorter than the default 20, to keep runs quick
}

In [ ]:
%matplotlib inline
import json
from pathlib import Path

from IPython.display import Image, display

from stem_agent.graphs.particle_graph import build_graph, mermaid, run
from stem_agent.nodes import common
from stem_agent.utils.plots import plot_graph


def report(state):
    print(f"run_status: {state['run_status']} | errors: {state.get('errors') or 'none'}")
    print(f"folder: {state['run']['run_dir']}")
    figure = Path(state["run"]["run_dir"]) / "summary.png"
    if figure.exists():
        display(Image(figure))

## The graph
Solid: the main path. Red dashed: the router's jump to `status_end` when a node fails or no particles are found. `status_end` and `save` always run.

In [ ]:
plot_graph(build_graph())

The same graph as mermaid text (made locally) — paste it into any mermaid viewer if you want an interactive diagram:

In [ ]:
print(mermaid())

## Run 1 · Normal
Expected: every node runs, `run_status: ok`, tuning z / C1 / A1, particles and spectra in `summary.json`, one-page `summary.png`.

In [ ]:
normal = await run({**BASE, "run_tag": "graph_normal"})
report(normal)

## Run 2 · No particles
`min_diameter_nm` is larger than any particle. Expected: `segment → status_end → save`, no EDS, `run_status: no_particles`.

In [ ]:
empty = await run({**BASE, "segmentation": {"min_diameter_nm": 50.0}, "run_tag": "graph_no_particles"})
report(empty)

## Run 3 · Failure
Test only: the 5th image fails (during `focus_z`). Expected: z restored, jump to `status_end` → `save`, `run_status: failed`, the cause in `errors`.

In [ ]:
original_acquire = common.acquire_haadf
count = {"images": 0}


async def failing_acquire(detector="haadf"):
    count["images"] += 1
    if count["images"] > 4:
        raise RuntimeError("test only: injected acquisition failure")
    return await original_acquire(detector)


common.acquire_haadf = failing_acquire
try:
    failed = await run({**BASE, "run_tag": "graph_failure"})
finally:
    common.acquire_haadf = original_acquire
z_start, z_end = failed["start"]["stage"][2], failed["end"]["stage"][2]
print(f"z at start {z_start * 1e6:+.3f} um, at end {z_end * 1e6:+.3f} um (restored: {z_start == z_end})")
report(failed)

## The three runs side by side

In [ ]:
for name, state in (("normal", normal), ("no particles", empty), ("failure", failed)):
    saved = json.loads(Path(state["summary_path"]).read_text())
    print(f"{name:<13} {saved['run_status']:<13} tuning {sorted(saved.get('tuning', {})) or '-'}  "
          f"particles {len(saved.get('particles') or [])}  spectra {len(saved.get('spectra') or [])}  "
          f"errors {len(saved['errors'])}")